# 面试题：如何不用现成架构手写 Decoder-only Transformer，并证明它利用了长程上下文？

## 面试回答主线

一个 Decoder block 通常由 causal multi-head self-attention、前馈网络、残差连接和 LayerNorm 组成。Pre-LN 结构在每个子层前归一化，残差路径保持恒等梯度，更利于深层训练。Attention 负责跨位置混合信息，逐位置 FFN 负责非线性特征变换；两者缺一不可。训练时把前缀位置的隐藏状态映射到词表 logits，再用交叉熵预测下一个 token。必须保证 causal mask、padding mask、position id 和 label shift 一致，否则离线 loss 可能因偷看答案而虚假变好。

## 真实案例：订单类型决定配送路线

八条流程的最近状态都是“仓库处理”，但下一事件取决于更早出现的商品类型。任务只预测配送事件，用来隔离 Transformer 是否真正读取长程前缀；这是教学型流程数据，不代表线上物流预测效果。

In [1]:
import math  # 导入平方根以实现注意力缩放和手写 GELU。
from collections import Counter  # 导入计数器以实现 bigram 基线。
import torch  # 导入 PyTorch 以构建并训练自定义 Transformer。
from torch import nn  # 导入基础模块、参数和 ModuleList 容器。
import torch.nn.functional as F  # 导入稳定的多类交叉熵损失。
torch.set_num_threads(1)  # 小张量教学实验固定单线程以减少并行调度开销。
flows = [  # 构造长程商品类型决定下一配送事件的订单流程。
    (["[BOS]", "生鲜订单", "已支付", "仓库处理"], "冷链配送"),  # 生鲜需要冷链路线。
    (["[BOS]", "冷冻食品", "已支付", "仓库处理"], "冷链配送"),  # 冷冻食品共享冷链标签。
    (["[BOS]", "鲜花订单", "已支付", "仓库处理"], "同城配送"),  # 鲜花要求短时同城配送。
    (["[BOS]", "蛋糕订单", "已支付", "仓库处理"], "同城配送"),  # 蛋糕同样走同城路线。
    (["[BOS]", "普通商品", "已支付", "仓库处理"], "标准快递"),  # 普通商品使用标准快递。
    (["[BOS]", "图书订单", "已支付", "仓库处理"], "标准快递"),  # 图书共享标准路线。
    (["[BOS]", "海外商品", "已支付", "仓库处理"], "国际配送"),  # 海外商品需要国际路线。
    (["[BOS]", "虚拟商品", "已支付", "仓库处理"], "即时到账"),  # 虚拟商品不走物理配送。
]  # 结束订单流程样本列表。
print("前缀流程                                      下一事件")  # 输出真实案例输入表标题。
for prefix, target in flows:  # 逐条展示完整前缀和监督目标。
    print(f"{' → '.join(prefix):<44} {target}")  # 输出流程事件链和下一个事件。

前缀流程                                      下一事件
[BOS] → 生鲜订单 → 已支付 → 仓库处理                    冷链配送
[BOS] → 冷冻食品 → 已支付 → 仓库处理                    冷链配送
[BOS] → 鲜花订单 → 已支付 → 仓库处理                    同城配送
[BOS] → 蛋糕订单 → 已支付 → 仓库处理                    同城配送
[BOS] → 普通商品 → 已支付 → 仓库处理                    标准快递
[BOS] → 图书订单 → 已支付 → 仓库处理                    标准快递
[BOS] → 海外商品 → 已支付 → 仓库处理                    国际配送
[BOS] → 虚拟商品 → 已支付 → 仓库处理                    即时到账


## Baseline（基线）：只看最近一个事件的 bigram

八条样本的最近事件全部是“仓库处理”，bigram 只能永远预测训练中频次最高且字典序最小的路线。它无法利用两个位置之前的商品类型，因此在少数类路线必然失败。

In [2]:
bigram_counts = Counter()  # 创建最近事件到下一事件的频次统计。
for prefix, target in flows:  # 遍历全部订单流程训练 bigram。
    bigram_counts[(prefix[-1], target)] += 1  # 只使用最后一个事件和目标构造有向 pair。
candidate_counts = Counter({target: count for (previous, target), count in bigram_counts.items() if previous == "仓库处理"})  # 汇总共同前态对应的路线次数。
baseline_target = sorted(candidate_counts.items(), key=lambda item: (-item[1], item[0]))[0][0]  # 用频次降序和字典序稳定选择唯一预测。
baseline_predictions = [baseline_target for _ in flows]  # 对所有相同最近事件输出同一个目标。
baseline_accuracy = sum(prediction == target for prediction, (_, target) in zip(baseline_predictions, flows)) / len(flows)  # 计算 bigram 逐样本准确率。
print("bigram 候选频次：", dict(sorted(candidate_counts.items())))  # 输出最近事件之后的目标分布。
print("固定预测：", baseline_target)  # 输出稳定 tie-break 选中的路线。
print(f"只看最近事件的准确率：{baseline_accuracy:.1%}")  # 输出后续 Transformer 的同数据基线。

bigram 候选频次： {'冷链配送': 2, '即时到账': 1, '同城配送': 2, '国际配送': 1, '标准快递': 2}
固定预测： 冷链配送
只看最近事件的准确率：25.0%


## 核心实现一：手写 LayerNorm、GELU 与 causal multi-head attention

没有调用 `nn.LayerNorm`、`nn.MultiheadAttention` 或现成 Transformer。LayerNorm 显式计算每个 token 的均值和方差；GELU 使用 erf 公式；Attention 显式拆头、生成上三角 mask、softmax 并合并各头。

In [3]:
class ManualLayerNorm(nn.Module):  # 定义沿最后一维归一化的可学习 LayerNorm。
    def __init__(self, dimension, epsilon=1e-5):  # 初始化缩放、偏置和数值稳定项。
        super().__init__()  # 注册基础模块状态。
        self.scale = nn.Parameter(torch.ones(dimension))  # 创建逐隐藏维度的可学习缩放。
        self.bias = nn.Parameter(torch.zeros(dimension))  # 创建逐隐藏维度的可学习偏置。
        self.epsilon = epsilon  # 保存防止除零的稳定常数。
    def forward(self, hidden):  # 对每个 token 的隐藏维度执行归一化。
        mean = hidden.mean(dim=-1, keepdim=True)  # 计算每个 token 的隐藏均值。
        variance = ((hidden - mean) ** 2).mean(dim=-1, keepdim=True)  # 计算总体方差而不是无偏样本方差。
        normalized = (hidden - mean) / torch.sqrt(variance + self.epsilon)  # 得到数值稳定的零均值单位方差表示。
        return normalized * self.scale + self.bias  # 应用可学习仿射变换。
def manual_gelu(hidden):  # 使用标准正态累积分布定义 GELU 激活。
    return 0.5 * hidden * (1.0 + torch.erf(hidden / math.sqrt(2.0)))  # 对输入逐元素执行平滑门控。
class CausalMultiHeadAttention(nn.Module):  # 定义不依赖现成 Attention 的 decoder 多头注意力。
    def __init__(self, model_dim, head_count):  # 初始化 Q/K/V 与输出投影。
        super().__init__()  # 注册基础模块状态。
        self.model_dim = model_dim  # 保存总隐藏维度。
        self.head_count = head_count  # 保存并行 head 数。
        self.head_dim = model_dim // head_count  # 计算每个 head 的维度。
        self.query_weight = nn.Parameter(torch.randn(model_dim, model_dim) * 0.08)  # 创建查询投影矩阵。
        self.key_weight = nn.Parameter(torch.randn(model_dim, model_dim) * 0.08)  # 创建键投影矩阵。
        self.value_weight = nn.Parameter(torch.randn(model_dim, model_dim) * 0.08)  # 创建值投影矩阵。
        self.output_weight = nn.Parameter(torch.randn(model_dim, model_dim) * 0.08)  # 创建多头输出投影矩阵。
    def split_heads(self, tensor):  # 把隐藏维拆分为多个注意力 head。
        batch_size, sequence_length, _ = tensor.shape  # 读取输入的批量与长度。
        return tensor.reshape(batch_size, sequence_length, self.head_count, self.head_dim).transpose(1, 2)  # 返回 batch、head、length、head_dim 顺序。
    def forward(self, hidden, causal=True):  # 计算可切换 causal 门禁的多头注意力。
        queries = self.split_heads(hidden @ self.query_weight)  # 投影并拆分查询张量。
        keys = self.split_heads(hidden @ self.key_weight)  # 投影并拆分键张量。
        values = self.split_heads(hidden @ self.value_weight)  # 投影并拆分值张量。
        scores = queries @ keys.transpose(-2, -1) / math.sqrt(self.head_dim)  # 计算缩放点积分数矩阵。
        if causal:  # decoder 正常路径必须屏蔽所有未来列。
            sequence_length = hidden.shape[1]  # 读取当前序列长度。
            future = torch.triu(torch.ones(sequence_length, sequence_length, dtype=torch.bool), diagonal=1)  # 创建严格上三角未来位置 mask。
            scores = scores.masked_fill(future[None, None, :, :], torch.finfo(scores.dtype).min)  # 把未来分数替换为浮点最小值。
        weights = torch.softmax(scores, dim=-1)  # 沿 key 维把分数归一化为注意力概率。
        head_outputs = weights @ values  # 在每个 head 内对值向量加权求和。
        merged = head_outputs.transpose(1, 2).reshape(hidden.shape[0], hidden.shape[1], self.model_dim)  # 合并多个 head 的子空间。
        return merged @ self.output_weight, weights  # 返回输出投影后的上下文和完整注意力矩阵。
torch.manual_seed(3)  # 固定机制演示的随机输入和参数。
norm_preview = ManualLayerNorm(6)  # 创建六维手写归一化层。
preview_hidden = torch.randn(2, 4, 6) * 3.0 + 2.0  # 构造均值和方差明显偏移的隐藏张量。
normalized_preview = norm_preview(preview_hidden)  # 执行手写 LayerNorm 前向传播。
print("归一化前每个 token 的均值：", [round(float(value), 4) for value in preview_hidden.mean(dim=-1)[0]])  # 输出第一条序列的原始均值。
print("归一化后每个 token 的均值：", [round(float(value), 6) for value in normalized_preview.mean(dim=-1)[0]])  # 输出手写归一化后的均值。
print("归一化后每个 token 的方差：", [round(float(value), 4) for value in normalized_preview.var(dim=-1, unbiased=False)[0]])  # 输出手写归一化后的总体方差。

归一化前每个 token 的均值： [1.9754, 1.8461, 3.6502, 1.6442]
归一化后每个 token 的均值： [-0.0, -0.0, 0.0, 0.0]
归一化后每个 token 的方差： [1.0, 1.0, 1.0, 1.0]


## 核心实现二：Pre-LN Block 与 Decoder 模型

每个 block 严格执行 `x = x + Attention(LN(x))`、`x = x + FFN(LN(x))`。模型把最后一个前缀位置“仓库处理”的隐藏状态映射到下一事件词表；若 Attention 能读取更早的商品类型，同一个当前位置就能产生不同路线。

In [4]:
class FeedForward(nn.Module):  # 定义逐 token 的两层前馈子网络。
    def __init__(self, model_dim, hidden_dim):  # 初始化扩展层与压缩层参数。
        super().__init__()  # 注册基础模块状态。
        self.first_weight = nn.Parameter(torch.randn(model_dim, hidden_dim) * 0.08)  # 创建隐藏维到扩展维的投影。
        self.first_bias = nn.Parameter(torch.zeros(hidden_dim))  # 创建扩展层偏置。
        self.second_weight = nn.Parameter(torch.randn(hidden_dim, model_dim) * 0.08)  # 创建扩展维回到模型维的投影。
        self.second_bias = nn.Parameter(torch.zeros(model_dim))  # 创建输出层偏置。
    def forward(self, hidden):  # 执行线性扩展、GELU 和线性压缩。
        expanded = hidden @ self.first_weight + self.first_bias  # 把每个 token 投影到更宽的特征空间。
        activated = manual_gelu(expanded)  # 使用手写 GELU 引入逐元素非线性。
        return activated @ self.second_weight + self.second_bias  # 压缩回模型隐藏维度。
class DecoderBlock(nn.Module):  # 定义包含两个 Pre-LN 残差子层的 decoder block。
    def __init__(self, model_dim, head_count, feedforward_dim):  # 初始化归一化、Attention 和 FFN。
        super().__init__()  # 注册基础模块状态。
        self.attention_norm = ManualLayerNorm(model_dim)  # 创建 Attention 前的手写 LayerNorm。
        self.attention = CausalMultiHeadAttention(model_dim, head_count)  # 创建手写 causal 多头注意力。
        self.feedforward_norm = ManualLayerNorm(model_dim)  # 创建 FFN 前的手写 LayerNorm。
        self.feedforward = FeedForward(model_dim, feedforward_dim)  # 创建逐位置前馈网络。
    def forward(self, hidden, causal=True):  # 执行完整 Pre-LN decoder block。
        attention_output, weights = self.attention(self.attention_norm(hidden), causal)  # 对归一化输入执行可切换 causal Attention。
        hidden = hidden + attention_output  # 通过第一条残差路径保留原表示并加入跨位置信息。
        hidden = hidden + self.feedforward(self.feedforward_norm(hidden))  # 通过第二条残差路径加入逐位置非线性特征。
        return hidden, weights  # 返回 block 输出与可解释注意力矩阵。
class TinyDecoderTransformer(nn.Module):  # 定义从 token id 到下一事件 logits 的完整 Decoder-only Transformer。
    def __init__(self, vocabulary_size, model_dim, head_count, layer_count, max_length):  # 初始化 embedding、block 堆叠、终层归一化和词表头。
        super().__init__()  # 注册基础模块状态。
        self.token_weight = nn.Parameter(torch.randn(vocabulary_size, model_dim) * 0.10)  # 创建可学习 token embedding 表。
        self.position_weight = nn.Parameter(torch.randn(max_length, model_dim) * 0.05)  # 创建可学习绝对位置表。
        self.blocks = nn.ModuleList([DecoderBlock(model_dim, head_count, model_dim * 3) for _ in range(layer_count)])  # 逐层构造自研 decoder blocks。
        self.final_norm = ManualLayerNorm(model_dim)  # 创建词表投影前的最终归一化。
        self.output_weight = nn.Parameter(torch.randn(vocabulary_size, model_dim) * 0.10)  # 创建独立的词表输出参数矩阵。
        self.output_bias = nn.Parameter(torch.zeros(vocabulary_size))  # 创建词表 logits 偏置。
    def forward(self, ids, causal=True):  # 从批量 token id 计算每个位置的下一 token logits。
        sequence_length = ids.shape[1]  # 读取当前前缀长度。
        hidden = self.token_weight[ids] + self.position_weight[:sequence_length].unsqueeze(0)  # 相加 token 与绝对位置表示。
        attention_layers = []  # 创建列表保存每层注意力供审计。
        for block in self.blocks:  # 按深度顺序执行每个 decoder block。
            hidden, weights = block(hidden, causal)  # 通过一个 Pre-LN Attention+FFN block。
            attention_layers.append(weights)  # 保存当前层完整注意力矩阵。
        normalized = self.final_norm(hidden)  # 对最终隐藏状态执行手写归一化。
        logits = normalized @ self.output_weight.transpose(0, 1) + self.output_bias  # 把每个位置映射到完整词表 logits。
        return logits, attention_layers, normalized  # 返回 logits、逐层权重和最终隐藏状态。
vocabulary = sorted({token for prefix, target in flows for token in prefix + [target]})  # 收集前缀与目标出现的完整事件词表。
token_to_id = {token: index for index, token in enumerate(vocabulary)}  # 创建确定性的事件 token id 映射。
input_ids = torch.tensor([[token_to_id[token] for token in prefix] for prefix, _ in flows], dtype=torch.long)  # 把四事件前缀转换为批量 id。
target_ids = torch.tensor([token_to_id[target] for _, target in flows], dtype=torch.long)  # 把下一配送事件转换为监督 id。
torch.manual_seed(53)  # 固定 Transformer 参数初始化。
transformer = TinyDecoderTransformer(len(vocabulary), 16, 2, 2, 6)  # 创建两层、两头、十六维 decoder 模型。
training_trace = []  # 保存关键轮次的 route loss 和准确率。
for step in range(1001):  # 执行完整前向、反向和手动 SGD 训练。
    all_logits, attention_layers, normalized = transformer(input_ids, causal=True)  # 计算每个前缀位置的词表分数。
    route_logits = all_logits[:, -1, :]  # 取“仓库处理”位置预测下一个事件。
    loss = F.cross_entropy(route_logits, target_ids)  # 计算多类下一事件交叉熵。
    loss.backward()  # 对 embedding、两层 block、LayerNorm 和词表头执行真实反向传播。
    with torch.no_grad():  # 参数更新过程不构建梯度图。
        for parameter in transformer.parameters():  # 逐个遍历全部自定义模块参数。
            parameter -= 0.06 * parameter.grad  # 使用固定学习率执行手动 SGD。
            parameter.grad.zero_()  # 清空当前梯度避免下一轮错误累积。
    if step in {0, 10, 50, 150, 400, 1000}:  # 保存能说明收敛过程的代表轮次。
        predictions = route_logits.detach().argmax(dim=1)  # 取得当前每条流程的最高分事件。
        accuracy = float((predictions == target_ids).to(torch.float32).mean())  # 计算当前下一事件准确率。
        training_trace.append((step, float(loss.detach()), accuracy))  # 保存轮次、损失和准确率。
print("轮次 | route交叉熵 | 准确率")  # 输出真实 Transformer 训练轨迹标题。
for step, loss_value, accuracy in training_trace:  # 逐个展示代表训练状态。
    print(f"{step:>4} | {loss_value:>11.4f} | {accuracy:>6.1%}")  # 输出轮次、损失和准确率。

轮次 | route交叉熵 | 准确率
   0 |      3.0525 |   0.0%
  10 |      1.7133 |  62.5%
  50 |      0.7983 |  75.0%
 150 |      0.0314 | 100.0%
 400 |      0.0071 | 100.0%
1000 |      0.0022 | 100.0%


## 结果表：相同最近事件，不同长程前缀

下面打印每条流程的预测与正确目标，并查看最后一层“仓库处理”位置对四个前缀位置的注意力。不同商品类型必须产生不同 logits，才能超过 bigram。

In [5]:
with torch.no_grad():  # 评估阶段关闭梯度图以生成确定性输出。
    final_logits, final_attention_layers, final_hidden = transformer(input_ids, causal=True)  # 对全部流程执行最终 causal 前向传播。
    route_probabilities = torch.softmax(final_logits[:, -1, :], dim=1)  # 把最后前缀位置 logits 转换为词表概率。
    transformer_prediction_ids = route_probabilities.argmax(dim=1)  # 取得每条流程的最高概率下一事件。
transformer_predictions = [vocabulary[index] for index in transformer_prediction_ids.tolist()]  # 把预测 id 还原为可读事件文本。
transformer_accuracy = sum(prediction == target for prediction, (_, target) in zip(transformer_predictions, flows)) / len(flows)  # 计算下一事件准确率。
print("商品类型     正确路线   bigram预测  Transformer预测  置信度  最后一层head0注意力")  # 输出逐流程结果表标题。
for index, (prefix, target) in enumerate(flows):  # 遍历每条订单流程和真实目标。
    confidence = float(route_probabilities[index, transformer_prediction_ids[index]])  # 取得预测类别对应概率。
    attention_row = final_attention_layers[-1][index, 0, -1]  # 取得最后层第零头在仓库位置的注意力行。
    attention_view = [round(float(value), 3) for value in attention_row]  # 格式化四个前缀位置的权重。
    print(f"{prefix[1]:<10} {target:<8} {baseline_target:<10} {transformer_predictions[index]:<15} {confidence:>6.3f}  {attention_view}")  # 输出目标、两种预测、概率和注意力。
print(f"准确率对照：bigram={baseline_accuracy:.1%}，手写 Transformer={transformer_accuracy:.1%}")  # 输出相同任务和指标下的核心对照。

商品类型     正确路线   bigram预测  Transformer预测  置信度  最后一层head0注意力
生鲜订单       冷链配送     冷链配送       冷链配送             0.999  [0.274, 0.265, 0.242, 0.219]
冷冻食品       冷链配送     冷链配送       冷链配送             0.999  [0.276, 0.262, 0.243, 0.22]
鲜花订单       同城配送     冷链配送       同城配送             0.998  [0.264, 0.255, 0.265, 0.216]
蛋糕订单       同城配送     冷链配送       同城配送             0.998  [0.265, 0.27, 0.254, 0.211]
普通商品       标准快递     冷链配送       标准快递             0.998  [0.236, 0.253, 0.263, 0.248]
图书订单       标准快递     冷链配送       标准快递             0.998  [0.232, 0.255, 0.265, 0.247]
海外商品       国际配送     冷链配送       国际配送             0.996  [0.248, 0.286, 0.25, 0.215]
虚拟商品       即时到账     冷链配送       即时到账             0.996  [0.274, 0.266, 0.23, 0.231]
准确率对照：bigram=25.0%，手写 Transformer=100.0%


## 结果解读

bigram 面对共同最近事件只能输出一个固定路线；两层 Decoder 通过 causal attention 把“生鲜/海外/虚拟”等更早 token 汇入仓库位置，再由 FFN 和词表头产生不同分布。Pre-LN 的均值/方差输出验证了归一化公式，训练轨迹则证明模型参数经过真实梯度更新，而不是预先写死路由表。

## 失败案例：teacher forcing 忘记 causal mask，标签会泄漏

把真实下一事件追加在前缀末尾，比较“仓库处理”位置的 logits。若关闭 causal mask，修改未来答案会改变当前预测；打开 mask 后，当前位置只能读取自己和更早前缀，两次 logits 必须一致。

In [6]:
original_full = torch.tensor([[token_to_id[token] for token in flows[0][0] + [flows[0][1]]]], dtype=torch.long)  # 构造包含正确未来路线的 teacher-forcing 序列。
changed_full = original_full.clone()  # 复制完整序列以保持所有前缀 token 不变。
changed_full[0, -1] = token_to_id["即时到账"]  # 只把未来答案替换为另一条合法路线。
with torch.no_grad():  # 泄漏诊断不需要构建梯度图。
    unmasked_original = transformer(original_full, causal=False)[0][:, 3, :]  # 错误地允许仓库位置读取正确未来答案。
    unmasked_changed = transformer(changed_full, causal=False)[0][:, 3, :]  # 在未来答案改变后重复无 mask 前向。
    causal_original = transformer(original_full, causal=True)[0][:, 3, :]  # 使用 causal mask 计算正确未来版本。
    causal_changed = transformer(changed_full, causal=True)[0][:, 3, :]  # 使用 causal mask 计算未来已替换版本。
unmasked_logit_shift = float((unmasked_original - unmasked_changed).abs().max())  # 量化无 mask 时未来标签对当前 logits 的污染。
causal_logit_shift = float((causal_original - causal_changed).abs().max())  # 量化 causal 修复后的前缀一致性。
causal_attention_row = transformer(original_full, causal=True)[1][0][0, 0, 3]  # 取得第一层第零头在仓库位置的完整权重行。
print("未来答案：", flows[0][1], "→ 即时到账")  # 输出反例中唯一被修改的未来 token。
print(f"未加 causal mask 的仓库位置最大 logit 漂移：{unmasked_logit_shift:.6f}")  # 展示 teacher-forcing 标签泄漏。
print(f"加 causal mask 后的仓库位置最大 logit 漂移：{causal_logit_shift:.6f}")  # 展示前缀预测恢复一致。
print("仓库位置的 causal attention：", [round(float(value), 4) for value in causal_attention_row])  # 展示未来路线列权重严格为零。

未来答案： 冷链配送 → 即时到账
未加 causal mask 的仓库位置最大 logit 漂移：4.108321
加 causal mask 后的仓库位置最大 logit 漂移：0.000000
仓库位置的 causal attention： [0.2567, 0.341, 0.1719, 0.2304, 0.0]


## 生产差距与落地清单

教学模型只优化一个位置、没有 dropout、KV cache、混合精度和分布式训练。完整语言模型需要对所有非 padding label 做正确 shift，并监控 token 加权 loss、困惑度、长程切片与生成质量；服务端要验证全量前向和逐 token KV cache logits 一致。FlashAttention 等内核只能优化计算，不能替你修复 mask、position offset 或数据泄漏。上线前必须加入不同长度、左/右 padding、全 mask、超长上下文和未来扰动测试。

## 最小回归测试

断言只保护归一化、长程效果和因果隔离；自研类定义、真实训练轨迹和逐流程预测才是架构复现的主体。

In [7]:
assert float(normalized_preview.mean(dim=-1).abs().max()) < 1e-5  # 验证手写 LayerNorm 产生近似零均值。
assert float((normalized_preview.var(dim=-1, unbiased=False) - 1.0).abs().max()) < 1e-4  # 验证手写 LayerNorm 产生近似单位总体方差。
assert training_trace[-1][1] < training_trace[0][1]  # 验证自研 Transformer 的真实反向传播降低交叉熵。
assert transformer_accuracy > baseline_accuracy  # 验证长程上下文模型优于只看最近事件的 bigram。
assert unmasked_logit_shift > 1e-5  # 固化关闭 causal mask 后未来标签污染当前 logits 的反例。
assert causal_logit_shift < 1e-7  # 验证 causal mask 让相同前缀的 logits 完全一致。
assert float(causal_attention_row[-1]) == 0.0  # 验证仓库位置对未来路线 token 的注意力严格为零。
print("最小回归测试通过：Pre-LN、长程路由学习和 teacher-forcing 因果隔离均符合预期。")  # 输出完整顺序执行成功的明确结论。

最小回归测试通过：Pre-LN、长程路由学习和 teacher-forcing 因果隔离均符合预期。
